<a href="https://colab.research.google.com/github/FatiBuuloloo/Image_Anomaly_Detection_on_MVTec_Dataset-mini_project_005/blob/main/Image_anomaly_detection_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anomalib

In [ ]:
!pip install onnxscript

In [ ]:
from anomalib.data import MVTec
from anomalib.models import Padim
from anomalib.engine import Engine
from anomalib.data import Folder
from anomalib.models import Patchcore
import matplotlib.pyplot as plt

In [ ]:
def denormalize(img_tensor):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min())
    return img
def show_results(predictions, num_samples=5):

    for i, pred in enumerate(predictions):
        if i >= num_samples: break
        try:
            anomaly_map = pred.anomaly_map[0].cpu().numpy()
            score = pred.pred_score[0].item()
            label_idx = pred.pred_label[0].item()
            img_original = denormalize(pred.image[0])
        except AttributeError:
            anomaly_map = pred['anomaly_map'][0].cpu().numpy()
            score = pred['pred_score'][0].item()
            label_idx = pred['pred_label'][0].item()
            img_original = denormalize(pred['image'][0])

        label = "ANOMALI (CACAT)" if label_idx else "NORMAL (BAGUS)"

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(img_original)
        axes[0].set_title(f"Original\n{label}", color='red' if label_idx else 'green')
        axes[0].axis('off')

        im = axes[1].imshow(anomaly_map, cmap='jet')
        axes[1].set_title(f"Heatmap\nScore: {score:.4f}")
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

        axes[2].imshow(img_original)
        axes[2].imshow(anomaly_map, cmap='jet', alpha=0.4)
        axes[2].set_title("Overlay Location")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

# Iterative ZIP Gdrive

In [ ]:
categories = [ "transistor", "wood", "zipper"]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
all_data = {}

for item in categories:
    path_zip = f"/content/drive/MyDrive/dataset/{item}.zip"
    print(f"Unzipping {item}")
    !unzip -oq "$path_zip" -d "/content/dataset/"
    dm = Folder(
        name=f"{item}_project",
        root=f"/content/dataset/{item}",
        normal_dir="train",
        abnormal_dir="test",
        normal_test_dir="test/good",
        eval_batch_size=1)
    all_data[item] = dm
    print(f"finish unzipping {item}")
    print()

# Transistor

In [ ]:
data_transistor = all_data["transistor"]
data_transistor.setup()

In [ ]:
model_transistor = Padim(backbone="resnet18")

In [ ]:
engine_transistor = Engine()
engine_transistor.fit(model=model_transistor, datamodule=data_transistor)

In [ ]:
test_transistor = engine_transistor.test(model=model_transistor, datamodule=data_transistor)
print(test_transistor)

In [ ]:
predictions_transistor = engine_transistor.predict(model=model_transistor,datamodule=data_transistor)
show_results(predictions_transistor,num_samples=5)

PatchCore Model

In [ ]:
model_transistor_2 = Patchcore(
    backbone="resnext50_32x4d",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1
)
engine_transistor_2 = Engine(
    enable_checkpointing=True,
    logger=True,
enable_progress_bar=False,
    devices=1
)
engine_transistor_2.fit(model=model_transistor_2, datamodule=data_transistor)

In [ ]:
test_transistor_2 = engine_transistor_2.test(model=model_transistor_2, datamodule=data_transistor)
print(test_transistor_2)

In [ ]:
predictions_transistor_2 = engine_transistor_2.predict(model=model_transistor_2,datamodule=data_transistor)
show_results(predictions_transistor_2,num_samples=5)